# 01. 프롬프트 엔지니어링 실습

---

## 학습 목표

LLM의 출력 품질을 높이는 **핵심 프롬프트 기법 6가지**를 실습합니다.  
같은 질문에 다른 기법을 적용해 결과 차이를 직접 확인합니다.

## 다루는 기법

| 기법 | 핵심 아이디어 | 언제 쓰나? |
|---|---|---|
| **Zero-shot** | 예시 없이 바로 질문 | 간단한 일반 질문 |
| **Few-shot** | 예시(샷)를 프롬프트에 포함 | 특정 형식·스타일 유도 |
| **Chain-of-Thought** | 단계별 추론 유도 | 복잡한 분석·판단 |
| **역할 프롬프트** | LLM에게 페르소나 부여 | 전문 도메인 답변 |
| **구조화된 출력** | 특정 형식(JSON 등) 강제 | 파싱이 필요한 경우 |
| **프롬프트 템플릿** | 변수로 재사용 가능한 프롬프트 | 반복 작업 자동화 |

## 흐름

```
기법 학습 → 코드 실습 → 같은 질문으로 비교 실험 → 핵심 정리
```

In [1]:
import os
import sys
from dotenv import load_dotenv
sys.path.append("..")  # 상위 폴더의 common_config.py를 import하기 위해 추가
from common_config import get_llm

load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm("gpt-5.4-mini", temperature=0)

import json

print("준비 완료")
print("실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'")

준비 완료
실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'


---

## 1. Zero-shot Prompting

**예시 없이** 바로 질문하는 가장 기본적인 방식입니다.  
LLM이 사전 학습 지식만으로 답변합니다.

```
사용자: [질문]
LLM:   [답변]
```

In [2]:
# ── Zero-shot: 예시 없이 바로 질문 ────────────────────────────
zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

# 체인 생성 및 실행
chain = zero_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Zero-shot 결과 ===")
print(result)

=== Zero-shot 결과 ===
짧게 말하면 **“전면 도입”은 무조건 찬성보다, 업종·직무에 따라 신중히 찬성**하는 게 맞습니다.  
재택근무는 분명 장점이 크지만, 전면 도입에는 부작용도 있습니다.

## 찬성할 만한 이유
- **생산성 향상 가능**: 이동시간이 줄고 집중이 잘 되는 사람도 많습니다.
- **우수 인재 확보**: 지역 제한 없이 채용할 수 있어 인력 풀이 넓어집니다.
- **비용 절감**: 사무실 임대, 운영비를 줄일 수 있습니다.
- **워라밸 개선**: 육아, 돌봄, 건강 등 개인 사정과 병행하기 좋습니다.
- **재난 대응력 강화**: 감염병, 기상 악화 등에도 업무 연속성이 높아집니다.

## 반대하거나 조심해야 할 이유
- **소통과 협업 약화**: 즉각적인 피드백, 우연한 아이디어 교류가 줄 수 있습니다.
- **성과 관리 어려움**: 결과 중심 문화가 없으면 관리만 복잡해집니다.
- **고립감·소속감 저하**: 특히 신입사원은 문화 적응과 학습이 어렵습니다.
- **업무 경계 붕괴**: 집과 일이 섞여 번아웃이 오기 쉽습니다.
- **보안·정보관리 이슈**: 자료 유출, 네트워크 보안 문제를 더 신경 써야 합니다.

## 결론
- **사무직, 개별 성과 측정이 쉬운 직무**라면 찬성 쪽이 강합니다.
- 하지만 **전면 도입**보다는  
  **하이브리드(재택+출근 병행)**가 현실적이고 안전한 경우가 많습니다.
- 핵심은 “집에서 일하느냐”가 아니라 **성과 측정, 소통 방식, 보안, 교육 체계가 갖춰졌느냐**입니다.

원하시면 제가 이 주제로  
1) **찬성 논리**, 2) **반대 논리**, 3) **토론용 입장문**  
형태로 바로 정리해드릴게요.


---

## 2. Few-shot Prompting

프롬프트 안에 **예시(shots)** 를 포함해서 원하는 출력 형식을 보여줍니다.  
LLM이 패턴을 학습해 같은 형식으로 답변합니다.

```
시스템: 예시1: Q→A, 예시2: Q→A  ← 패턴 학습
사용자: [새 질문]
LLM:   [예시와 같은 형식의 답변]  ← 패턴 따라함
```

In [3]:
# ── Few-shot: 예시 2개를 포함해서 출력 형식을 유도 ────────────
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 예시와 똑같은 형식으로 답변하세요.

예시 1)
질문: 주 4일 근무제를 도입해야 할까요?
답변:
✅ 찬성 근거
- 직원 삶의 질 향상으로 생산성 증가
- 채용 경쟁력 강화

❌ 반대 근거
- 업무량 미감소 시 오히려 번아웃 위험
- 서비스 연속성 유지 어려움

💡 결론: 업종·직무별로 차등 적용하는 파일럿 테스트를 권장합니다.

예시 2)
질문: 사내 카페테리아를 없애야 할까요?
답변:
✅ 찬성 근거
- 운영 비용 절감
- 외부 다양한 식사 옵션 활용 가능

❌ 반대 근거
- 직원 식사 복지 감소
- 점심 시간 이탈로 협업 기회 감소

💡 결론: 직원 수요 조사 후 결정하되, 식대 지원으로 보완을 고려하세요."""),
    ("human", "질문: {question}")
])

# 체인 생성 및 실행
chain = few_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Few-shot 결과 ===")
print(result)

=== Few-shot 결과 ===
✅ 찬성 근거
- 출퇴근 시간 절감으로 업무 만족도와 집중도 향상
- 지역 제한 없이 우수 인재 채용 가능
- 사무실 운영비 절감 효과 기대

❌ 반대 근거
- 대면 소통 감소로 협업·창의성 저하 가능
- 근무 관리와 성과 평가가 어려워질 수 있음
- 업무·생활 경계가 흐려져 번아웃 위험 증가

💡 결론: 전면 도입보다는 직무 특성에 따라 선택형 재택근무를 운영하는 것이 바람직합니다.


---

## 3. Chain-of-Thought (CoT) Prompting

LLM에게 **단계별로 생각하도록** 유도합니다.  
복잡한 추론, 계획, 분석이 필요한 질문에 효과적입니다.

```
기본:  질문 → 바로 답변 (shallow)
CoT:  질문 → 단계1 → 단계2 → 단계3 → 결론 (deep)
```

In [4]:
# ── Chain-of-Thought: 단계별 사고 과정 명시 ───────────────────
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """질문에 답하기 전에 반드시 다음 4단계로 생각하고, 각 단계를 명시하세요.

[1단계: 문제 파악] 무엇을 묻고 있는가?
[2단계: 이해관계자 분석] 누가 영향을 받는가?
[3단계: 장단점 검토] 도입/미도입 시 각각 어떤 결과가?
[4단계: 최종 판단] 근거를 바탕으로 한 결론은?"""),
    ("human", "{question}")
])

# 체인 생성 및 실행
chain = cot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Chain-of-Thought 결과 ===")
print(result)

=== Chain-of-Thought 결과 ===
[1단계: 문제 파악] 무엇을 묻고 있는가?  
“재택근무 전면 도입에 찬성할지”에 대한 판단을 묻고 있습니다. 즉, 회사가 전 직원 대상의 상시 재택근무 체제로 가는 것이 좋은지 여부를 검토하는 질문입니다.

[2단계: 이해관계자 분석] 누가 영향을 받는가?  
- **직원**: 출퇴근 부담 감소, 일과 삶의 균형 개선 가능  
- **관리자/팀장**: 성과 관리, 협업 방식 재설계 필요  
- **회사 경영진**: 생산성, 비용, 조직문화, 인재 유지에 영향  
- **고객/협력사**: 응대 속도, 서비스 품질 변화 가능  
- **신입/주니어 직원**: 학습·멘토링·소속감 형성에 영향

[3단계: 장단점 검토] 도입/미도입 시 각각 어떤 결과가?  

**도입할 경우 장점**  
- 통근 시간 절감으로 직원 만족도 상승  
- 지역 제약이 줄어 인재 채용 범위 확대  
- 사무실 비용 절감 가능  
- 집중 업무는 효율이 오를 수 있음

**도입할 경우 단점**  
- 협업, 즉흥적 소통, 창의적 논의가 약해질 수 있음  
- 팀 결속과 조직문화 유지가 어려울 수 있음  
- 성과 관리가 불명확하면 생산성 편차가 커질 수 있음  
- 신입 교육, 온보딩, 멘토링이 약해질 수 있음  
- 보안, 개인정보, 장비 관리 이슈가 생길 수 있음

**미도입할 경우 장점**  
- 대면 협업과 빠른 의사소통이 쉬움  
- 조직문화와 팀워크를 유지하기 유리함  
- 관리 체계가 익숙해 운영이 안정적임  
- 신입 육성과 현장 학습에 유리함

**미도입할 경우 단점**  
- 통근 스트레스와 시간 손실이 큼  
- 유연성이 낮아 이직률 상승 가능  
- 우수 인재 유치 경쟁에서 불리할 수 있음  
- 사무실 비용이 계속 발생함

[4단계: 최종 판단] 근거를 바탕으로 한 결론은?  
**무조건 전면 도입에 찬성하기보다는, “업무 특성에 따라 선택적·혼합형 재택근무”에 찬성하는 편이 더 합리적입니다.**  
이유는 재택근

---

## 4. 역할(Role) 프롬프트

LLM에게 **특정 전문가 페르소나**를 부여합니다.  
같은 질문이라도 역할에 따라 완전히 다른 관점으로 답변합니다.

```
역할 없음:  중립적·일반적 답변
HR 전문가:  사람 중심의 조직 관점
CFO:       비용·효율 중심의 재무 관점
```

In [5]:
# ── 역할 프롬프트: 같은 질문, 다른 역할 ──────────────────────
def make_role_chain(role_description: str):
    """역할 설명을 받아 전문가 체인을 생성한다"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", role_description),
        ("human", "{question}")
    ])
    chain = prompt | llm | StrOutputParser()
    return chain


# 역할 1: HR 디렉터
hr_chain = make_role_chain("""
당신은 15년 경력의 HR 디렉터입니다.
직원 복지, 조직 문화, 인재 유지에 최우선 가치를 둡니다.
답변은 구성원 관점에서 3~4문장으로 핵심만 짚어주세요.
""")

# 역할 2: 재무이사(CFO)
cfo_chain = make_role_chain("""
당신은 10년 경력의 재무이사(CFO)입니다.
모든 결정을 비용-효익 분석으로 접근합니다.
답변은 수치 근거를 중심으로 3~4문장으로 핵심만 짚어주세요.
""")

question = "재택근무 전면 도입, 찬성해야 할까요?"

print("=== HR 디렉터 관점 ===")
print(hr_chain.invoke({"question": question}))

print("\n=== CFO 관점 ===")
print(cfo_chain.invoke({"question": question}))

=== HR 디렉터 관점 ===
전면 재택은 구성원의 자율성과 몰입도를 높일 수 있어 분명 매력적이지만, 모든 조직에 무조건 정답은 아닙니다.  
업무의 협업 빈도, 신입 온보딩, 보안, 팀 소속감 유지가 잘 설계되지 않으면 오히려 피로감과 고립감이 커질 수 있습니다.  
그래서 “찬성/반대”보다, 직무 특성과 팀 성숙도에 맞는 하이브리드부터 검증해보는 게 구성원 만족도와 생산성을 함께 지키는 현실적 선택입니다.  
구성원 입장에서는 출퇴근 부담이 줄어드는 장점이 크니, 전면 도입을 원한다면 최소한 소통 규칙, 성과 기준, 복지 보완책까지 함께 제안하는 것이 좋습니다.

=== CFO 관점 ===
결론부터 말하면 **전면 도입은 조건부 찬성, 무조건 찬성은 비추천**입니다.  
재택 전면화는 **오피스 임대·관리비를 1인당 연간 100만~300만 원 수준까지 절감**할 수 있지만, 반대로 **협업 저하, 보안 투자, 관리 공백**으로 생산성이 **5~15%만 흔들려도 절감분이 상쇄**될 수 있습니다.  
따라서 **개발·디자인·문서업무처럼 성과 측정이 명확한 직무는 재택 확대**, **영업·신입교육·대면 협업이 중요한 조직은 하이브리드**가 비용 대비 효율이 높습니다.  
즉, **전면 도입보다 “직무별 선택형 재택”이 재무적으로 더 합리적**입니다.


---

## 5. 구조화된 출력 (Structured Output)

LLM의 출력을 **JSON 등 특정 형식으로 강제**합니다.  
후처리(파싱, DB 저장, API 응답)가 필요할 때 사용합니다.

```
일반 출력: 자연어 문장 → 파싱 불가
구조화:   JSON/YAML → 자동 파싱 가능 → 시스템 연동 가능
```

In [6]:
# ── 구조화된 출력: JSON 형식 강제 ────────────────────────────
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """반드시 아래 JSON 형식으로만 답변하세요. 다른 텍스트는 절대 포함하지 마세요.

  - 출력형식
  "topic": "주제",
  "pros": ["찬성 근거1", "찬성 근거2", "찬성 근거3"],
  "cons": ["반대 근거1", "반대 근거2"],
  "recommendation": "최종 권고사항",
  "confidence": 확신도(1~10)
"""),
    ("human", "다음 주제를 분석해줘: {question}")
])

# 체인 생성 및 실행
chain = structured_prompt | llm | StrOutputParser()
raw_output = chain.invoke({"question": "재택근무 전면 도입"})

print("=== LLM 원본 출력 ===")
print(raw_output)

# JSON 파싱
print("\n=== 파싱 후 활용 ===")
try:
    data = json.loads(raw_output)
    print(f"주제     : {data['topic']}")
    print(f"확신도   : {data['confidence']}/10")
    print(f"권고사항 : {data['recommendation']}")
    print(f"찬성 근거: {', '.join(data['pros'])}")
except json.JSONDecodeError as e:
    print(f"파싱 실패: {e}")
    print("→ 팁: 모델이 JSON을 못 지키면 출력에서 JSON 부분만 추출하거나 재시도 로직을 추가하세요.")

=== LLM 원본 출력 ===
{"topic":"재택근무 전면 도입","pros":["출퇴근 시간이 사라져 직원의 시간 효율과 삶의 만족도가 높아질 수 있음","사무실 임대료, 관리비 등 고정비를 줄여 기업 비용 절감에 도움이 됨","지역 제한 없이 인재를 채용할 수 있어 채용 범위와 조직 다양성이 확대됨"],"cons":["협업과 소통이 비대면 중심으로 바뀌어 팀 결속과 즉각적인 의사결정이 약해질 수 있음","업무 성과 관리와 보안 통제가 어려워질 수 있음","직무 특성상 대면이 필요한 업무에는 적용이 어렵고 구성원 간 형평성 이슈가 생길 수 있음"],"recommendation":"전면 도입은 모든 조직에 일괄적으로 적합하지 않으므로, 직무별로 재택 가능성을 분류한 뒤 하이브리드 방식과 병행하는 것이 가장 현실적입니다. 특히 성과 관리 체계, 보안 규정, 협업 도구를 충분히 갖춘 뒤 단계적으로 확대하는 것을 권고합니다.","confidence":9}

=== 파싱 후 활용 ===
주제     : 재택근무 전면 도입
확신도   : 9/10
권고사항 : 전면 도입은 모든 조직에 일괄적으로 적합하지 않으므로, 직무별로 재택 가능성을 분류한 뒤 하이브리드 방식과 병행하는 것이 가장 현실적입니다. 특히 성과 관리 체계, 보안 규정, 협업 도구를 충분히 갖춘 뒤 단계적으로 확대하는 것을 권고합니다.
찬성 근거: 출퇴근 시간이 사라져 직원의 시간 효율과 삶의 만족도가 높아질 수 있음, 사무실 임대료, 관리비 등 고정비를 줄여 기업 비용 절감에 도움이 됨, 지역 제한 없이 인재를 채용할 수 있어 채용 범위와 조직 다양성이 확대됨


---

## 6. 프롬프트 템플릿 (재사용 가능한 팩토리)

**변수화된 프롬프트**를 만들어 다양한 도메인에 재사용합니다.  
같은 구조의 프롬프트를 반복해서 만들 때 효율적입니다.

```python
# 나쁜 예: 비슷한 프롬프트를 계속 복붙
legal_prompt   = "당신은 변호사입니다... {question}"
medical_prompt = "당신은 의사입니다... {question}"

# 좋은 예: 팩토리 함수로 생성
make_expert_chain(domain="법률", style="간결하게")
make_expert_chain(domain="의료", style="쉽게 설명")
```

In [7]:
# ── 프롬프트 템플릿 팩토리 ────────────────────────────────────
def make_analyst_chain(domain: str, perspective: str, output_format: str):
    """
    재사용 가능한 분석 체인을 생성한다.

    Args:
        domain      : 전문 도메인 (예: '경영', '기술', '심리')
        perspective : 분석 관점 (예: '비용 중심', '사람 중심')
        output_format: 출력 형식 지시문
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""당신은 {domain} 분야 전문 분석가입니다.
{perspective} 관점에서 분석하세요.

출력 형식:
{output_format}"""),
        ("human", "{question}")
    ])
    
    # 체인 생성
    chain = prompt | llm | StrOutputParser()
    return chain


# 경영 분석 체인
biz_chain = make_analyst_chain(
    domain="경영",
    perspective="ROI와 생산성 중심",
    output_format="[핵심 지표] → [예상 효과] → [실행 권고] (각 2~3줄)"
)

# 조직심리 분석 체인
psych_chain = make_analyst_chain(
    domain="조직심리",
    perspective="구성원 동기·몰입 중심",
    output_format="[심리적 영향] → [리스크] → [권고사항] (각 2~3줄)"
)

question = "재택근무 전면 도입, 찬성해야 할까요?"

print("=== 경영 분석 관점 ===")
print(biz_chain.invoke({"question": question}))

print("\n=== 조직심리 관점 ===")
print(psych_chain.invoke({"question": question}))

=== 경영 분석 관점 ===
[핵심 지표] → 재택근무 전면 도입의 ROI는 “고정비 절감”과 “인건비 대비 생산성”에서 갈립니다. 사무실 축소로 임대·관리비를 줄일 수 있지만, 협업 비용·보안 비용·관리 공수는 늘어날 수 있습니다.  
생산성은 개인 업무형 직무에선 상승 가능성이 있으나, 부서 간 조정이 많은 조직은 오히려 지연이 생길 수 있습니다.

[예상 효과] → 잘 설계하면 출퇴근 시간 절감, 이직률 감소, 채용 경쟁력 상승으로 성과가 개선됩니다. 반대로 전면 도입을 서두르면 커뮤니케이션 손실, 온보딩 약화, 성과 관리 불명확으로 매출/품질이 떨어질 수 있습니다.  
즉, “전면 찬성”보다 “직무별 적합성에 따라 선택적 도입”이 ROI 측면에서 더 안전합니다.

[실행 권고] → 전면 도입 찬반보다 먼저 3가지로 판단하세요: ① 직무가 비대면 독립형인지, ② 성과가 수치화 가능한지, ③ 협업 빈도가 낮은지. 이 3가지가 높으면 찬성, 낮으면 혼합형이 유리합니다.  
추천안은 “하이브리드 기본 + 완전 재택 가능 직무만 예외 적용”입니다. 3개월 파일럿으로 생산성, 이직률, 사무실 비용을 측정한 뒤 확대 여부를 결정하세요.

=== 조직심리 관점 ===
[심리적 영향] → 재택근무 전면 도입은 자율성과 통제감이 높아져 내적 동기와 업무 몰입을 강화할 수 있습니다. 특히 집중이 필요한 업무에서는 방해가 줄어 성과 만족도가 올라갈 가능성이 큽니다. 다만 사회적 연결감이 약해지면 소속감과 정서적 에너지가 떨어질 수 있습니다.

[리스크] → 구성원 간 정보 비대칭과 고립감이 커지면 협업 동기, 학습, 혁신 행동이 약화될 수 있습니다. 또한 성과가 보이지 않는 상황에서 평가 공정성에 대한 불신이나 “보이지 않는 사람”에 대한 불안이 생길 수 있습니다. 신입·저경력자는 학습과 관계 형성이 더 어려워집니다.

[권고사항] → 전면 찬반보다 “직무·구성원별 선택형 하이브리드”가 심리적으로 더 안정적입니다. 재택 비중을 높이더라도 정기적 대면 협업일, 명확한 성과기

---

## 7. 비교 실험: 같은 질문, 다른 기법

지금까지 배운 6가지 기법을 **같은 질문**에 적용해 결과를 비교합니다.  
답변의 구조, 깊이, 형식이 얼마나 다른지 확인하세요.

In [8]:
# ── 비교 실험: 기법별 답변 길이 & 구조 비교 ──────────────────
QUESTION = "재택근무 전면 도입, 찬성해야 할까요?"

# 각 기법의 체인 목록
techniques = [
    ("Zero-shot",         zero_shot_prompt),
    ("Few-shot",          few_shot_prompt),
    ("Chain-of-Thought",  cot_prompt),
]

print(f"질문: {QUESTION}")
print("=" * 60)

results = {}
for name, prompt in techniques:
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": QUESTION})
    results[name] = answer

# 결과 요약 출력
print(f"\n{'기법':<20} {'글자수':>8} {'줄 수':>6}")
print("-" * 38)
for name, answer in results.items():
    print(f"{name:<20} {len(answer):>8,} {len(answer.splitlines()):>6}")

질문: 재택근무 전면 도입, 찬성해야 할까요?

기법                        글자수    줄 수
--------------------------------------
Zero-shot               1,115     55
Few-shot                  206     11
Chain-of-Thought        1,249     46


In [9]:
# ── 답변 전체 출력 ────────────────────────────────────────────
for name, answer in results.items():
    print(f"\n{'='*60}")
    print(f"[{name}]")
    print('='*60)
    # 처음 400자만 출력 (너무 길면 ...으로 줄임)
    preview = answer[:400]
    print(preview)
    if len(answer) > 400:
        print(f"\n... (총 {len(answer):,}자 중 400자 미리보기)")


[Zero-shot]
결론부터 말하면, **“전면 도입”에 무조건 찬성하기보다는 조건부 찬성**이 더 현실적입니다.  
재택근무는 분명 장점이 크지만, 모든 회사·직무에 항상 최선은 아니기 때문입니다.

## 찬성할 만한 이유
1. **출퇴근 시간 절감**
   - 직원 만족도와 삶의 질이 좋아집니다.
   - 피로와 스트레스가 줄어 생산성이 오를 수 있습니다.

2. **비용 절감**
   - 회사는 사무실 임대, 관리비, 부대비용을 줄일 수 있습니다.
   - 직원도 교통비, 식비 부담이 줄어듭니다.

3. **일과 삶의 균형 개선**
   - 육아, 돌봄, 개인 일정과 병행이 쉬워집니다.
   - 인재 유치와 유지에 도움이 됩니다.

4. **채용 범위 확대**
   - 지역에 상관없이 인재를 뽑을 수 있습니다.
   - 

... (총 1,115자 중 400자 미리보기)

[Few-shot]
✅ 찬성 근거
- 출퇴근 시간 절감으로 업무 효율과 만족도 향상
- 지역과 무관하게 우수 인재 채용 가능
- 사무실 운영비 절감 효과 기대

❌ 반대 근거
- 협업·소통 저하로 업무 속도와 팀 결속 약화
- 보안 관리와 성과 측정이 어려워질 수 있음
- 직무에 따라 재택 적합성이 크게 다름

💡 결론: 전면 도입보다는 직무별·부서별 선택적 재택근무가 더 현실적입니다.

[Chain-of-Thought]
[1단계: 문제 파악] 무엇을 묻고 있는가?  
재택근무를 **전면 도입하는 것에 찬성해야 하는지**를 묻고 있습니다. 즉, 효율성, 조직문화, 성과, 관리 가능성까지 포함해 **도입 여부의 타당성**을 판단해야 합니다.

[2단계: 이해관계자 분석] 누가 영향을 받는가?  
- **직원**: 통근 부담 감소, 일·생활 균형 개선, 고립감 증가 가능  
- **관리자/리더**: 성과관리 방식 변화, 소통·협업 관리 부담 증가  
- **회사/조직**: 비용 절감 가능, 조직문화 약화 리스크 존재  
- **고객/협력사**: 응대 속도와 품질이 달라질 수 있음  
-

---

## 핵심 정리

```python
# 프롬프트 엔지니어링 선택 가이드

# 1. Zero-shot: 빠른 테스트, 일반 질문
chain = ChatPromptTemplate.from_messages([("human", "{q}")]) | llm | StrOutputParser()

# 2. Few-shot: 특정 형식 유도, 스타일 일관성
#    → 시스템 프롬프트에 예시 2~3개 포함

# 3. Chain-of-Thought: 복잡한 추론, 다단계 분석
#    → "단계별로 생각하세요" 지시문 추가

# 4. 역할 프롬프트: 전문 도메인, 특정 관점 필요 시
#    → 구체적일수록 좋음 ("전문가" < "10년 경력 변호사")

# 5. 구조화 출력: 파싱·연동이 필요한 경우
#    → JSON 형식 + "다른 텍스트 포함 금지" 명시

# 6. 팩토리 패턴: 동일 구조 반복 시
def make_chain(domain, style):
    prompt = ChatPromptTemplate.from_messages([...])
    return prompt | llm | StrOutputParser()
```

| 기법 | 최적 상황 | 주의사항 |
|---|---|---|
| Zero-shot | 범용 질문, 빠른 프로토타입 | 형식 보장 없음 |
| Few-shot | 특정 형식·스타일 필요 | 예시가 너무 많으면 토큰 낭비 |
| CoT | 복잡한 추론·판단 | 응답이 길어짐 |
| 역할 | 전문 도메인 | 구체적일수록 효과적 |
| 구조화 출력 | API 연동, 파싱 필요 | 100% 보장 안 됨 (검증 로직 필요) |
| 팩토리 | 반복 사용 패턴 | 과도한 추상화 주의 |
